In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipelineV2 import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [12]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 22 teams with confirmed lineups


### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTIONV2.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTIONV2.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTIONV2.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_listV2.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")
print(f"Loaded features: {len(features)}")

Loaded models with calibration factor: 4.4
Loaded features: 56


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Darius Garland,Over,15.5,-137,2025-11-29,2025-11-28T08:55:35Z
1,Underdog,player_points,Darius Garland,Under,15.5,-137,2025-11-29,2025-11-28T08:55:35Z
2,Underdog,player_points,Donovan Mitchell,Over,29.5,-137,2025-11-29,2025-11-28T08:55:35Z
3,Underdog,player_points,Donovan Mitchell,Under,29.5,-137,2025-11-29,2025-11-28T08:55:35Z
4,Underdog,player_points,Kristaps Porzingis,Over,18.5,-137,2025-11-29,2025-11-28T08:55:35Z


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10, use_bias_adjustment=True)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 126 players...
Processing 117 players...
Generated 6513 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
3503,Pascal Siakam,Devin Booker,25.5,24.5,-103,-110,31.71,30.52,0.845,0.868,over,over,1,115.75,0.579,High,Med
3030,Cade Cunningham,Lauri Markkanen,28.5,26.5,-125,-125,34.20,31.69,0.823,0.831,over,over,1,100.93,0.505,High,Med
6326,LeBron James,James Harden,21.5,26.5,-137,-118,16.88,32.04,0.819,0.810,under,over,1,95.20,0.476,Med,High
5141,Cameron Johnson,Precious Achiuwa,14.5,7.5,-126,-106,10.39,3.92,0.805,0.799,under,under,0,89.02,0.445,Low,Low
3788,Bennedict Mathurin,De'Aaron Fox,23.5,24.5,-108,-128,27.81,28.91,0.774,0.772,over,over,1,75.73,0.379,Med,Med
1919,LaMelo Ball,T.J. McConnell,18.5,12.5,-114,-118,22.77,8.80,0.764,0.772,over,under,0,73.25,0.366,Med,Low
1778,Day'Ron Sharpe,Isaiah Collier,7.5,8.5,-115,-130,4.23,5.58,0.763,0.762,under,under,0,70.89,0.354,Low,Low
392,Jalen Johnson,Bruce Brown,20.5,8.5,-121,-118,24.37,5.36,0.761,0.762,over,under,0,70.47,0.352,Med,Low
3149,Franz Wagner,Cedric Coward,23.5,12.5,-124,-115,27.43,16.13,0.754,0.753,over,over,0,67.04,0.335,Med,Med
124,Donovan Mitchell,Jared McCain,29.5,12.5,-111,-104,33.48,9.16,0.751,0.751,over,under,0,65.82,0.329,Med,Low


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 143 players...
Processing 129 players...
Generated 7923 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,RAW PREDICTION 1,RAW PREDICTION 2,BIAS ADJUSTMENT 1,BIAS ADJUSTMENT 2,CONFIDENCE 1,CONFIDENCE 2,REASON 1,REASON 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2366,Drake Powell,Cameron Johnson,9.5,14.5,-130,-126,6.84,9.39,1.43,-1.00,HIGH,MEDIUM,Q30‑Q40: mild overprediction (~1.4 pts),Q60‑Q70: underprediction (+1.0 pts),5.41,10.39,under,under,0.823,0.805,0.6494,0.258,0.247,0.341,94.82,0.474,1,4.40,4.79,Low,Low,"(0.0, 14.0)","(1.0, 19.8)",0.05,0,94.8
2505,Jalen Wilson,Precious Achiuwa,7.5,7.5,-106,-106,6.43,6.39,2.47,2.47,HIGH,HIGH,Q20‑Q30: consistent overprediction (~2.5 pts),Q20‑Q30: consistent overprediction (~2.5 pts),3.96,3.92,under,under,0.797,0.799,0.6242,0.283,0.284,0.364,87.25,0.436,0,4.25,4.27,Low,Low,"(0.0, 12.3)","(0.0, 12.3)",0.05,0,87.3
5308,Cam Whitmore,Austin Reaves,11.5,21.5,-118,-125,7.85,22.12,0.43,-4.16,HIGH,LOW,Q40‑Q50: near neutral (‑0.4 pts),Q80‑Q90: large underprediction (+4.2 pts),7.42,26.28,under,over,0.795,0.794,0.6182,0.253,0.238,0.323,85.46,0.427,1,4.96,5.83,Low,Med,"(0.0, 17.1)","(14.9, 37.7)",0.05,0,85.5
2751,LaMelo Ball,Bennedict Mathurin,18.5,23.5,-114,-108,20.65,20.02,-2.12,-7.79,MEDIUM,LOW,Q70‑Q80: notable underprediction (+2.1 pts),>Q90: massive underprediction (+7.8 pts),22.77,27.81,over,over,0.764,0.774,0.5795,0.231,0.255,0.308,73.84,0.369,1,5.94,5.72,Med,Med,"(11.1, 34.4)","(16.6, 39.0)",0.05,0,73.8
5236,T.J. McConnell,LeBron James,12.5,20.5,-118,-108,8.24,12.72,-0.56,-4.16,HIGH,LOW,Q50‑Q60: slight underprediction (+0.6 pts),Q80‑Q90: large underprediction (+4.2 pts),8.80,16.88,under,under,0.772,0.763,0.5770,0.230,0.244,0.301,73.09,0.365,0,4.97,5.06,Low,Med,"(0.0, 18.5)","(7.0, 26.8)",0.05,0,73.1
6659,Bruce Brown,Isaiah Collier,8.5,8.5,-118,-130,6.79,7.01,1.43,1.43,HIGH,HIGH,Q30‑Q40: mild overprediction (~1.4 pts),Q30‑Q40: mild overprediction (~1.4 pts),5.36,5.58,under,under,0.762,0.762,0.5688,0.220,0.197,0.269,70.64,0.353,0,4.42,4.10,Low,Low,"(0.0, 14.0)","(0.0, 13.6)",0.05,0,70.6
28,Jalen Johnson,Franz Wagner,20.5,23.5,-121,-124,20.21,19.64,-4.16,-7.79,LOW,LOW,Q80‑Q90: large underprediction (+4.2 pts),>Q90: massive underprediction (+7.8 pts),24.37,27.43,over,over,0.761,0.754,0.5629,0.214,0.201,0.266,68.88,0.344,0,5.45,5.70,Med,Med,"(13.7, 35.1)","(16.3, 38.6)",0.05,0,68.9
1984,Jared McCain,Cedric Coward,12.5,12.5,-104,-115,8.60,15.57,-0.56,-0.56,HIGH,HIGH,Q50‑Q60: slight underprediction (+0.6 pts),Q50‑Q60: slight underprediction (+0.6 pts),9.16,16.13,under,over,0.751,0.753,0.5545,0.242,0.218,0.287,66.36,0.332,0,4.92,5.30,Low,Med,"(0.0, 18.8)","(5.7, 26.5)",0.05,0,66.4
6807,Dillon Brooks,Kyle Filipowski,17.0,9.5,-137,-137,18.38,7.80,-2.12,1.43,MEDIUM,HIGH,Q70‑Q80: notable underprediction (+2.1 pts),Q30‑Q40: mild overprediction (~1.4 pts),20.50,6.37,over,under,0.736,0.747,0.5386,0.158,0.169,0.211,61.57,0.308,0,5.55,4.72,Med,Low,"(9.6, 31.4)","(0.0, 15.6)",0.05,0,61.6
2607,Coby White,Walter Clayton Jr.,23.5,5.5,-106,-132,19.24,6.62,-7.79,3.71,LOW,MEDIUM,>Q90: massive underprediction (+7.8 pts),Q10-Q20: heavy overprediction (~3.7 pts),27.03,2.91,over,under,0.729,0.733,0.5238,0.215,0.164,0.237,57.15,0.286,0,5.78,4.18,Med,Low,"(15.7, 38.4)","(0.0, 11.1)",0.05,0,57.1


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 112 players...
Processing 103 players...
Generated 155856 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
141229,Cameron Johnson,Precious Achiuwa,LeBron James,14.5,7.5,21.5,10.39,3.92,16.88,0.805,0.799,0.819,under,under,under,0,184.49,0.369,Low,Low,Med
58027,Day'Ron Sharpe,LaMelo Ball,Bennedict Mathurin,7.5,18.5,23.5,4.23,22.77,27.81,0.763,0.764,0.774,under,over,over,0,143.51,0.287,Low,Med,Med
119418,T.J. McConnell,Bruce Brown,Isaiah Collier,12.5,8.5,8.5,8.80,5.36,5.58,0.772,0.762,0.762,under,under,under,0,141.88,0.284,Low,Low,Low
11018,Jalen Johnson,Franz Wagner,Cedric Coward,20.5,23.5,12.5,24.37,27.43,16.13,0.761,0.754,0.753,over,over,over,0,133.60,0.267,Med,Med,Med
50438,Jared McCain,Kyle Filipowski,Austin Reaves,12.5,9.5,22.5,9.16,6.37,26.28,0.751,0.747,0.742,under,under,over,0,124.71,0.249,Low,Low,Med


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 143 players...
Processing 129 players...
Generated 308203 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
131399,Drake Powell,Cameron Johnson,Precious Achiuwa,9.5,14.5,7.5,5.41,10.39,3.92,0.823,0.805,0.799,under,under,under,0,185.86,0.372,Low,Low,Low
135756,Jalen Wilson,Cam Whitmore,Austin Reaves,7.5,11.5,21.5,3.96,7.42,26.28,0.797,0.795,0.794,under,under,over,0,171.59,0.343,Low,Low,Med
149366,LaMelo Ball,Bennedict Mathurin,LeBron James,18.5,23.5,20.5,22.77,27.81,16.88,0.764,0.774,0.763,over,over,under,0,143.58,0.287,Med,Med,Med
249925,T.J. McConnell,Bruce Brown,Isaiah Collier,12.5,8.5,8.5,8.80,5.36,5.58,0.772,0.762,0.762,under,under,under,0,141.88,0.284,Low,Low,Low
3022,Jalen Johnson,Franz Wagner,Cedric Coward,20.5,23.5,12.5,24.37,27.43,16.13,0.761,0.754,0.753,over,over,over,0,133.60,0.267,Med,Med,Med


In [ ]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)
# len(df)

105